# To run in tmux:
 ```bash
 tmux new -s train
 ```
 ```bash
 papermill src/train.ipynb notebooks/train_out.ipynb --cwd src --log-output --autosave-cell-every 60
 ```

In [ ]:
import numpy as np
import torch
import cv2
from torch.utils.data import Dataset
import json
from pathlib import Path

PROJECT_ROOT = Path("../")
DATASET_PATH = PROJECT_ROOT / "datasets" / "depth_v4_custom_triplets"

class DepthHandDataset(Dataset):
    def __init__(self, npz_paths, frames_per_clip=60, crop_size=224, bbox_padding=1.25):
        self.npz_paths = list(npz_paths)
        self.frames_per_clip = frames_per_clip
        self.total_frames = len(self.npz_paths) * self.frames_per_clip
        self.crop_size  = crop_size
        self.bbox_padding = bbox_padding
        
        dataset_dir = Path(self.npz_paths[0]).parent
        cam_config_path = dataset_dir / "cam_config.json"
        
        # TODO: modify build_dataset.py to copy cam_config from triplets folder
        # to dataset folder automatically
        if not cam_config_path.exists():
            raise FileNotFoundError(f"Camera config not found: {cam_config_path}")
        
        with open(cam_config_path, "r", encoding="utf-8") as f:
            cam_config = json.load(f)
            
        intrinsics = cam_config["intrinsics"]
        self.fx = float(intrinsics[0][0])
        self.cx = float(intrinsics[0][2])
        self.cy = float(intrinsics[1][2])
        
        print(f">>> [Dataset] Loaded camera config: fx={self.fx:.2f}, cx={self.cx:.1f}, cy={self.cy:.1f}")
        
    def __len__(self):
        return self.total_frames
    
    def __getitem__(self, idx):
        clip_idx = idx // self.frames_per_clip 
        frame_idx = idx % self.frames_per_clip
        clip_path = self.npz_paths[clip_idx]
        
        with np.load(clip_path) as data:
            depth = data["depth"][frame_idx]        # (H, W) -> (480, 848)
            joints = data["joints_3d"][frame_idx]   # (21, 3)
            cx, cy, size = data["bbox"][frame_idx]
            
            cx_noise = np.random.uniform(-15.0, 15.0)
            cy_noise = np.random.uniform(-15.0, 15.0)
            scale_noise = np.random.uniform(0.9, 1.1)
            
            cx += cx_noise
            cy += cy_noise
            size *= scale_noise
            
            # bbox params
            H, W = depth.shape
            pad_size = size * self.bbox_padding
            half = pad_size / 2.0
            x1 = max(0, int(round(cx - half)))
            y1 = max(0, int(round(cy - half)))
            x2 = min(W, int(round(cx + half)))
            y2 = min(H, int(round(cy + half)))
            
            # apply crop
            crop = depth[y1:y2, x1:x2]
            crop_resized = cv2.resize(crop, (self.crop_size, self.crop_size), interpolation=cv2.INTER_NEAREST)
            X = torch.from_numpy(crop_resized).float().unsqueeze(0)
            
            # joint positions in crop frame
            u_full = (self.fx * joints[:, 0] / joints[:, 2]) + self.cx
            v_full = (self.fx * joints[:, 1] / joints[:, 2]) + self.cy
            
            scale_factor = self.crop_size / float(pad_size)
            u_crop = (u_full - x1) * scale_factor
            v_crop = (v_full - y1) * scale_factor
            z_metric = joints[:, 2]
            
            # (x,y) from pixels to meters (to match z scale)
            u_center = self.crop_size / 2.0
            v_center = self.crop_size / 2.0
            fx_crop = self.fx * scale_factor
            
            x_rel_m = (u_crop - u_center) * z_metric / fx_crop
            y_rel_m = (v_crop - v_center) * z_metric / fx_crop
            
            target = np.stack([x_rel_m, y_rel_m, z_metric], axis=-1)
            y = torch.from_numpy(target).float()
            
            return X, y

In [ ]:
import random
from torch.utils.data import DataLoader
SPLIT_RATIO = 0.9

random.seed(42)
npz_clips = sorted(list(DATASET_PATH.glob("*.npz")))
clips_shuffled = npz_clips.copy()
random.shuffle(clips_shuffled)

split_idx = int(SPLIT_RATIO * len(clips_shuffled))
train_files = clips_shuffled[:split_idx]
val_files = clips_shuffled[split_idx:]

print(f"Clips for TRAIN: {len(train_files)} ({len(train_files)*60} frames)")
print(f"Clips for VAL:   {len(val_files)} ({len(val_files)*60} frames)")

train_dataset = DepthHandDataset(train_files)
val_dataset = DepthHandDataset(val_files)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

In [ ]:
import sys
from pathlib import Path
if str(Path("..").resolve()) not in sys.path:
    sys.path.insert(0, str(Path("..").resolve()))
if str(Path(".").resolve()) not in sys.path:
    sys.path.insert(0, str(Path(".").resolve()))

try:
    from src.models import HandPoseIPR
except ImportError:
    from models import HandPoseIPR


In [ ]:
import torch.nn as nn
def mpjpe(y_hat, y_gt):
    diff = y_hat - y_gt # (Batch, 21, 3)
    dist = torch.linalg.vector_norm(diff, dim=-1) 
    
    return dist.mean() * 1000 # mjpe in mm
 
criterion = nn.MSELoss()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net = HandPoseIPR().to(device)

In [ ]:
# check corrupted files
'''
import numpy as np
from tqdm import tqdm

npz_files = sorted(list(DATASET_PATH.glob("*.npz")))
corrupted_files = []

for p in tqdm(npz_files):
    try:
        with np.load(p) as data:
            _ = data["depth"].shape
            _ = data["joints_3d"].shape
    except Exception as e:
        print(f"\n>>> File corrupted: {p.name} ({e})")
        corrupted_files.append(p)

if corrupted_files:
    print(f">>> Found corrupted files: {len(corrupted_files)}")
    for f in corrupted_files:
        print(f" -> {f}")
else:
    print(">>> No corrupted files.")
'''

In [ ]:
from tqdm import tqdm

EPOCHS = 10

optimizer = torch.optim.AdamW(net.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
best_val_mpjpe = float("inf")

for epoch in range(EPOCHS):
    # TRAIN
    net.train()
    train_loss_total = 0.0
    train_mpjpe_total = 0.0

    train_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for x, y in train_bar:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad()
        y_hat = net(x)
        loss = criterion(y_hat, y)
        loss.backward()
        optimizer.step()
        train_loss_total += loss.item()
        with torch.no_grad():
            batch_mpjpe = mpjpe(y_hat, y).item()
            train_mpjpe_total += batch_mpjpe

        train_bar.set_postfix({
            "loss": f"{loss.item():.5f}",
            "mpjpe": f"{batch_mpjpe:.1f} mm"
        })

    train_bar.close()   
    train_loss_avg = train_loss_total / len(train_dataloader)
    train_mpjpe_avg = train_mpjpe_total / len(train_dataloader)

    # VAL
    net.eval()
    val_loss_total = 0.0
    val_mpjpe_total = 0.0

    val_bar = tqdm(val_dataloader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]")
    with torch.no_grad():
        for x, y in val_bar:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            y_hat = net(x)
            loss = criterion(y_hat, y)

            val_loss_total += loss.item()
            val_mpjpe_total += mpjpe(y_hat, y).item()

    val_bar.close()
    val_loss_avg = val_loss_total / len(val_dataloader)
    val_mpjpe_avg = val_mpjpe_total / len(val_dataloader)

    scheduler.step()

    # EPOCH SUMMARY
    print(f"\n>>> Epoch {epoch+1}")
    print(f"Train Loss: {train_loss_avg:.5f} | Train MPJPE: {train_mpjpe_avg:.2f} mm")
    print(f"Val Loss:   {val_loss_avg:.5f} | Val MPJPE:   {val_mpjpe_avg:.2f} mm")
    print(f"Current LR: {scheduler.get_last_lr()[0]:.2e}")

    if val_mpjpe_avg < best_val_mpjpe:
        best_val_mpjpe = val_mpjpe_avg
        torch.save(net.state_dict(), "best_hand_model.pth")
        print(f"New best model saved! (MPJPE: {best_val_mpjpe:.2f} mm)")
    print("\n")